In [32]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn import linear_model
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from math import sqrt
df = pd.read_excel("../../clean_coast_data.xlsx")

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25898 entries, 0 to 25897
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   coast_num           25898 non-null  int64         
 1   Area                25898 non-null  object        
 2   Distance from gaza  24095 non-null  float64       
 3   center              25898 non-null  int64         
 4   city rank           25898 non-null  int64         
 5   bathing_season      25898 non-null  int64         
 6   summer_vaction      25898 non-null  int64         
 7   winter              25898 non-null  int64         
 8   river               25898 non-null  int64         
 9   military            25898 non-null  int64         
 10  nature              25898 non-null  int64         
 11  coast_name          25898 non-null  object        
 12  coast_desc          25354 non-null  object        
 13  Date test           25898 non-null  datetime64

In [34]:
df['Area'].unique()

array(['North', 'Center', 'Eilat', 'South'], dtype=object)

In [35]:
# using get_dummies on education column
encoded = pd.get_dummies(df.Area)
# Concatenate the dummies to original dataframe
merged_data = pd.concat([df, encoded], axis=1)
# dropping the original column which was not encoded
merged_data.drop(columns='Area', inplace=True)

In [36]:
merged_data.drop(columns=['coast_name','coast_desc','city','check station','key','coast_num','center'], inplace=True)

Quick & Dirty

In [51]:
merged_data['Date test num'] = merged_data['Date test'].view(np.int64)
without_nulls = merged_data.dropna()

without_nulls.sort_values('Date test')
y = without_nulls[['test_result','Date test']]
X = without_nulls.drop(['test_result','test_result_num'],axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size = 0.2)
X_train_scaler = MinMaxScaler()
X_test_scaler = MinMaxScaler()
X_train_scaled = X_train_scaler.fit_transform(X_train.drop('Date test',axis=1))
X_test_scaled = X_test_scaler.fit_transform(X_test.drop('Date test',axis=1))

mlr = linear_model.LinearRegression()
mlr.fit(X_train_scaled,y_train.test_result)

prediction = mlr.predict(X_train_scaled)
resTrain =  pd.DataFrame(
    data={
        'Prediction':prediction, 
        'Actual': y_train.test_result.values    },
        index=y_train['Date test']
)
prediction = mlr.predict(X_test_scaled)
resTest = pd.DataFrame(
    data={
        'Prediction':prediction,
        'Actual': y_test.test_result.values    },
        index=y_test['Date test']
)

print('rmse train',sqrt(mean_squared_error(resTrain.Prediction, resTrain.Actual)))
print('std train',np.std(resTrain.Actual))
print('Sample')
print(resTrain.sample(10))
print()

print('rmse test',sqrt(mean_squared_error(resTest.Prediction, resTest.Actual)))
print('std test',np.std(resTest.Actual))
print('Sample')
print(resTest.sample(10))
print()

(25898, 32) (21987, 29)
rmse train 0.9580245290196995
std train 0.9875581616850897
Sample
            Prediction  Actual
Date test                     
2012-09-23    3.599609       5
2019-03-29    3.798828       5
2015-08-24    3.632812       3
2012-05-07    3.445312       4
2020-11-14    4.095703       4
2008-12-15    3.753906       4
2021-08-31    3.761719       4
2017-06-20    3.500000       3
2012-08-01    3.730469       5
2012-02-26    3.376953       3

rmse test 2694289386.1911387
std test 1.0043944505309557
Sample
              Prediction  Actual
Date test                       
2013-12-02 -6.921545e+08       4
2010-02-14 -9.209359e+08       4
2012-04-22 -2.101276e+09       1
2020-10-29  3.279103e+09       2
2016-05-02  9.247298e+08       2
2022-12-31 -3.826530e+09       4
2015-03-25 -2.804400e+09       3
2020-07-30  3.849075e+09       5
2010-06-06 -9.212347e+08       4
2020-04-20 -1.078280e+09       2



with univariant feature selection

In [57]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression
from sklearn.metrics import r2_score

X = without_nulls.drop(['Date test','test_result','test_result_num'],axis=1)
y= without_nulls[['test_result']]

f_reg_selector = SelectKBest(f_regression, k=9)
f_reg_selector.fit(X, y)
f_reg_support = f_reg_selector.get_support()
f_reg_feature = X.loc[:,f_reg_support].columns.to_list()

uniX = X.loc[:,f_reg_support]
# 
X_train, X_test, y_train, y_test = train_test_split(uniX, y, shuffle=True, test_size = 0.2)
X_train_scaler = MinMaxScaler()
X_test_scaler = MinMaxScaler()
X_train_scaled = X_train_scaler.fit_transform(X_train)
X_test_scaled = X_test_scaler.fit_transform(X_test)

mlr = linear_model.LinearRegression()
mlr.fit(X_train_scaled,y_train.test_result)

prediction = mlr.predict(X_train_scaled)
resTrain =  pd.DataFrame(
    data={
        'Prediction':prediction, 
        'Actual': y_train.test_result.values    },
)
prediction = mlr.predict(X_test_scaled)
resTest = pd.DataFrame(
    data={
        'Prediction':prediction,
        'Actual': y_test.test_result.values    },
)

print('rmse train',sqrt(mean_squared_error(resTrain.Prediction, resTrain.Actual)))
print('std train',np.std(resTrain.Actual))
print('r2 train',r2_score(resTrain.Actual, resTrain.Prediction))
print('')
print('rmse test',sqrt(mean_squared_error(resTest.Prediction, resTest.Actual)))
print('std test',np.std(resTest.Actual))
print('r2 test',r2_score(resTest.Actual, resTest.Prediction))


rmse train 0.9613825203957748
std train 0.9895258159180326
r2 train 0.05607348613770502

rmse test 0.9710627074469687
std test 0.9966200694007352
r2 test 0.05063045740371175


c:\Users\liavc\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:72: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)
c:\Users\liavc\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:302: RuntimeWarning: invalid value encountered in true_divide
  corr /= X_norms
